---
# Dudley Qualifications — Data Cleaning

---
## Objectives

- Inspect the qualifications dataset and its geographical coverage.
- Check for missing values, duplicates and incorrect data types.
- Remove any non-data rows and document cleaning decisions.
- Save and verify a cleaned copy for later analysis.

## Inputs

- Raw dataset: `raw_datasets/dudley_qualifications.csv`
- Source: Office for National Statistics (ONS), Census 2021,
  accessed via Nomis.

  ## Outputs

- Cleaned dataset:
  `cleaned_datasets/dudley_qualifications_cleaned.csv`
- Documented data quality checks and cleaning decisions.

## Additional Comments

- The original raw CSV will remain unchanged.
- Population coverage, units and geographical identifiers
  will be checked before cleaning.
- Missing values and duplicates will be investigated before
  deciding how to handle them.
- Further changes and combining datasets will follow after
  initial cleaning of all five datasets.
- ChatGPT assists with code and documentation; outputs are
  reviewed before findings are recorded.




---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [2]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Dudley employment barriers\\jupyter_notebooks'

Move to the parent folder only when the current directory is
jupyter_notebooks. This prevents repeated runs from moving
outside the project folder.

In [3]:
# Move up only when running from the notebooks folder.
if os.path.basename(os.getcwd()) == "jupyter_notebooks":
    os.chdir("..")

print(os.getcwd())

c:\Dudley employment barriers


Confirm the new current directory

In [4]:
current_dir = os.getcwd()
current_dir

'c:\\Dudley employment barriers'

### Working Directory Findings

The working directory is the Dudley employment barriers project
folder. Relative paths can now access raw_datasets and
cleaned_datasets.

---

## 1. Import Libraries

- os: manage the working directory and output folders.
- pandas: load, inspect, clean and save the qualifications dataset.

The os library was also used in the working-directory setup above.

In [5]:
# Import the libraries needed for this cleaning notebook.
import os
import pandas as pd

print("Libraries imported successfully.")

Libraries imported successfully.


### Findings

Both libraries imported successfully. The notebook is ready
to load and inspect the qualifications dataset.

---

## 2. Inspect the Raw Dataset

Read the first ten lines of the qualifications CSV to identify
the title, population coverage, units and column headings.

This will establish whether introductory lines need to be skipped
when loading the full dataset. No changes will be made to the file.

## Ethical Considerations When Inspecting the Raw Qualifications Dataset

Inspecting the first ten lines of the raw qualifications CSV is an essential ethical step in the data‑cleaning process.  
Because this dataset describes real communities in Dudley, including their highest qualification levels, it is important to understand the file structure before loading it into pandas. This protects against misinterpretation and ensures responsible handling of sensitive socio‑economic information.

### Why Ethical Inspection Matters

The introductory lines of the file often contain:
- Titles  
- Population coverage  
- Units  
- Copyright notes  
- Metadata  
- Non‑data text  

These lines help establish:
- **Data provenance** — confirming the dataset comes from a trusted source (ONS/Nomis).  
- **Context** — understanding what the dataset measures and who it represents.  
- **Integrity** — identifying whether non‑data rows need to be skipped to avoid corrupting the dataset.

Understanding this context is crucial for ethical analysis, because qualification levels are strongly linked to employment opportunities, health outcomes, and long‑term socio‑economic disadvantage.

### Ethical Risks if Raw Inspection Is Ignored

If the raw file is loaded without inspection, several ethical risks arise:

- **Misinterpreting population coverage**  
  Analysts may assume the dataset covers all adults when it may only cover specific age groups.

- **Incorrect handling of metadata**  
  Non‑data rows could be mistaken for real records, distorting counts and misrepresenting communities.

- **Loss of transparency**  
  Without documenting the raw structure, users cannot verify that the dataset was loaded responsibly.

- **Misleading downstream analysis**  
  Incorrect column headings or skipped lines may lead to inaccurate dashboards or unfair comparisons between LSOAs.

### Ethical Safeguards Used in This Step

This inspection process supports ethical data handling by:

- **Displaying raw lines exactly as they appear**  
  Ensures transparency and allows manual verification.

- **Avoiding any modification to the raw file**  
  Preserves the original dataset for auditing and accountability.

- **Checking for blank lines and metadata**  
  Helps determine whether introductory rows must be skipped when loading the full dataset.

- **Documenting population coverage and units**  
  Prevents misinterpretation of qualification levels in later analysis.

### Why This Matters for Responsible Interpretation

Qualification data is sensitive and can influence how communities are perceived.  
Incorrect handling of the raw file can lead to:

- Misleading conclusions about educational disadvantage  
- Stigmatisation of neighbourhoods  
- Incorrect policy insights  
- Loss of trust in the analysis  

By inspecting the raw dataset carefully and transparently, this project ensures that Dudley’s qualification data is handled **fairly, accurately, and respectfully**, supporting ethical and responsible analysis throughout the workflow.



In [6]:
# Set the path to the raw qualifications dataset.
raw_path = "raw_datasets/dudley_qualifications.csv"

# Display the first ten lines with line numbers, including blanks.
with open(raw_path, encoding="utf-8-sig") as file:
    for line_number in range(1, 11):
        line = file.readline()
        if not line:
            break
        print(f"{line_number}: {line.rstrip()}")

1: 
2: "TS067 - Highest level of qualification"
3: "ONS Crown Copyright Reserved [from Nomis on 14 October 2026]"
4: "Population :","All usual residents aged 16 years and over"
5: "Units      :","Persons"
6: "Date       :","2021"
7: 
8: "2021 super output area - lower layer","mnemonic","Total: All usual residents aged 16 years and over","No qualifications","Level 1 and entry level qualifications","Level 2 qualifications","Apprenticeship","Level 3 qualifications","Level 4 qualifications or above","Other qualifications"
9: 
10: "Dudley 001A","E01009763",1637,436,183,289,102,286,296,45


### Findings from the Raw File Inspection

- The dataset is TS067: Highest level of qualification,
  from Census 2021.
- It covers all usual residents aged 16 years and over.
- Values represent counts of persons.
- The geography is 2021 Lower Layer Super Output Areas (LSOAs).
- Column headings appear on line 8, so the first seven lines
  will be skipped when loading the CSV.
- Area names and codes are already separate: the first column
  contains the area name, and mnemonic contains its code.
- There are 10 columns: two area identifiers, a population total
  and seven qualification categories.
- The first visible record is Dudley 001A, code E01009763.


---

### 2.1 Load and Inspect All Records

Load the qualifications CSV, skipping the seven lines before
the column headings.

Display every row and column to inspect the complete dataset
and identify any footer notes or empty records.

## Ethical Considerations When Loading the Qualifications Dataset

Loading the qualifications dataset is not just a technical step — it carries important **ethical responsibilities**, because this data describes real residents of Dudley and their highest qualification levels.  
Ensuring that the dataset is loaded correctly protects against misinterpretation, supports fairness, and maintains transparency throughout the analysis.

### Why Ethical Care Is Needed When Loading the Data

The raw file contains seven introductory lines before the column headings.  
These lines include:

- Dataset title  
- Population coverage  
- Units  
- Copyright information  
- Metadata  
- Non‑data text  

Skipping these lines is essential to avoid accidentally treating metadata as real records.  
Incorrect loading could distort counts, misrepresent communities, or lead to unfair comparisons between neighbourhoods.

### Ethical Risks if Loading Is Done Incorrectly

If the dataset is loaded without skipping the metadata lines, several ethical risks arise:

- **Non‑data rows may be interpreted as real LSOA records**  
  This could inflate population totals or qualification counts.

- **Column headings may be misaligned**  
  Leading to incorrect interpretation of qualification categories.

- **Misleading downstream analysis**  
  Dashboards or reports may show inaccurate values, affecting how communities are perceived.

- **Loss of transparency**  
  Users may not understand why certain rows appear corrupted or inconsistent.

### Ethical Safeguards Used in This Step

This loading process includes several responsible practices:

- **Skipping the first seven lines**  
  Ensures that only true column headings and data rows are loaded.

- **Preserving the raw file unchanged**  
  Maintains auditability and supports ethical data governance.

- **Displaying the dataset dimensions and first rows**  
  Allows manual verification that the dataset was loaded correctly.

- **Using UTF‑8‑SIG encoding**  
  Prevents hidden characters from corrupting column names or values.

### Why This Matters for Responsible Interpretation

Qualification levels are sensitive socio‑economic indicators.  
Incorrect loading can lead to:

- Misrepresentation of educational disadvantage  
- Stigmatisation of neighbourhoods  
- Incorrect policy insights  
- Loss of trust in the analysis  

By loading the dataset carefully and transparently, this project ensures that Dudley’s qualification data is handled **fairly, accurately, and respectfully**, supporting ethical and responsible analysis throughout the workflow.



In [7]:
# Developed with assistance from ChatGPT.

# Load the CSV using line 8 as the column headings.
df = pd.read_csv(
    raw_path,
    skiprows=7,
    encoding="utf-8-sig"
)

# Display the dimensions and first five rows.
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
display(df.head())

Rows: 208, Columns: 10


,2021 super output area - lower layer,mnemonic,Total: All usual residents aged 16 years and over,No qualifications,Level 1 and entry level qualifications,Level 2 qualifications,Apprenticeship,Level 3 qualifications,Level 4 qualifications or above,Other qualifications
0,Dudley 001A,E01009763,1637.0,436.0,183.0,289.0,102.0,286.0,296.0,45.0
1,Dudley 001B,E01009764,1266.0,306.0,132.0,187.0,98.0,214.0,279.0,50.0
2,Dudley 001C,E01009767,1265.0,358.0,146.0,220.0,73.0,181.0,238.0,49.0
3,Dudley 001D,E01009769,1428.0,467.0,157.0,244.0,88.0,211.0,221.0,40.0
4,Dudley 001E,E01009770,1215.0,390.0,152.0,188.0,72.0,181.0,198.0,34.0


### Initial inspection findings

- The raw qualifications dataset contains **208 rows and 10 columns**.
- The first five rows show Dudley neighbourhoods, with an **LSOA name** and **unique area code** recorded in separate columns.
- The dataset provides counts of residents aged **16 years and over**, grouped by their highest qualification level.
- Qualification categories range from **no qualifications** to **Level 4 qualifications or above**, alongside apprenticeships and other qualifications.
- Numeric counts currently display as decimals. Missing values and data types will be checked during cleaning.
- These figures are counts rather than percentages. Comparing qualification levels between neighbourhoods will require accounting for differences in population size.

---

## 3. Check missing values and final rows

Inspect missing values and the final five rows to identify any blank
records or source notes included in the CSV.

## Ethical Considerations When Checking Missing Values and Inspecting Final Rows

This step examines the qualifications dataset for missing values and inspects the final rows for blank records or source notes.  
Although this appears to be a routine technical check, it carries important **ethical responsibilities**, because qualification data reflects real educational outcomes for residents across Dudley’s neighbourhoods.

### Why Ethical Inspection Matters

Qualification levels are sensitive socio‑economic indicators.  
They influence how communities are understood in terms of:

- Access to employment  
- Long‑term economic opportunity  
- Educational disadvantage  
- Social mobility  

Ensuring the dataset is complete and free from structural issues is essential for **fair and responsible analysis**.

### Ethical Risks if Missing Values Are Ignored

If missing values or footer notes are not identified and handled correctly, several ethical risks arise:

- **Misrepresenting neighbourhoods**  
  Missing qualification counts could make some areas appear more disadvantaged than they are.

- **Inflating or deflating totals**  
  Blank rows or source notes may distort population counts or qualification distributions.

- **Incorrect comparisons**  
  Incomplete data may lead to unfair comparisons between LSOAs.

- **Misleading dashboards**  
  Visualisations may show inaccurate qualification levels, affecting how communities are perceived.

- **Loss of transparency**  
  Users may not understand why certain values appear inconsistent or incomplete.

### Ethical Safeguards Used in This Step

This inspection process includes several responsible practices:

- **Column‑level missing value checks**  
  Ensures that every qualification category is complete before analysis.

- **Displaying the final five rows**  
  Allows manual verification of any footer notes, blank records, or metadata included at the end of the file.

- **Preserving the raw dataset unchanged**  
  Maintains auditability and supports ethical data governance.

- **Clear reporting of missing values**  
  Supports transparency and accountability in the cleaning process.

### Why This Matters for Responsible Interpretation

Qualification data is often used to:

- Identify educational disadvantage  
- Support local policy decisions  
- Allocate resources  
- Understand long‑term employment barriers  

Incorrect handling of missing values or footer notes can lead to:

- Misleading conclusions  
- Stigmatisation of neighbourhoods  
- Incorrect policy insights  
- Loss of trust in the analysis  

By checking missing values and inspecting final rows carefully, this project ensures that Dudley’s qualification data is handled **fairly, accurately, and respectfully**, supporting ethical and responsible analysis throughout the workflow.



In [9]:
# Developed with assistance from ChatGPT.

# Check missing values in each column.
print("MISSING VALUES:")
print(df.isna().sum())

# Inspect the final five rows for blank records or source notes.
print("\nFINAL FIVE ROWS:")
display(df.tail())

MISSING VALUES:
2021 super output area - lower layer                 5
mnemonic                                             1
Total: All usual residents aged 16 years and over    5
No qualifications                                    5
Level 1 and entry level qualifications               5
Level 2 qualifications                               5
Apprenticeship                                       5
Level 3 qualifications                               5
Level 4 qualifications or above                      5
Other qualifications                                 5
dtype: int64

FINAL FIVE ROWS:


,2021 super output area - lower layer,mnemonic,Total: All usual residents aged 16 years and over,No qualifications,Level 1 and entry level qualifications,Level 2 qualifications,Apprenticeship,Level 3 qualifications,Level 4 qualifications or above,Other qualifications
203,NaN,In order to protect against disclosure of pers...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
204,NaN,have been swapped between different geographic...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
205,NaN,by small amounts. Small counts at the lowest g...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
206,NaN,affected.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
207,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Findings from the missing-value checks

The missing values are confined to the final five rows. Four contain
source notes about disclosure protection, and the last row is blank.
None represents a neighbourhood, so these are not missing
qualification counts that need to be filled in.

Removing these five rows should leave **203 neighbourhood records**.
The source notes will remain available in the original CSV.

---

## 4. Remove non-data rows

Keep rows containing a valid English LSOA code in the `mnemonic`
column. These codes begin with `E01` followed by six digits.

This will exclude the four source-note rows and the blank row.
Create a separate dataframe called `df_clean` to preserve the
original loaded data, then reset its row index.

Check the number of rows removed and confirm whether any missing
values remain. The original CSV retains the disclosure-control notes.

## Ethical Considerations When Removing Non‑Data Rows

This step filters the qualifications dataset to retain only valid English LSOA records and removes footer notes, metadata lines, and blank rows.  
Although this is a technical cleaning operation, it carries important **ethical responsibilities**, because qualification data reflects real educational outcomes for residents across Dudley’s neighbourhoods.

### Why Ethical Care Is Needed Here

The dataset includes:
- 203 genuine LSOA records  
- Four disclosure‑control note rows  
- One blank row  

Removing these rows is necessary for accurate analysis, but it must be done **transparently and cautiously** to avoid unintentionally excluding valid neighbourhoods or misrepresenting communities.

Qualification data is sensitive because it relates to:
- Educational disadvantage  
- Access to employment  
- Social mobility  
- Long‑term economic opportunity  

Any mistake in filtering can unfairly distort how communities are perceived.

### Ethical Risks if Filtering Is Done Incorrectly

If non‑data rows are not removed correctly, several ethical risks arise:

- **Accidental removal of real LSOAs**  
  Over‑strict patterns or incorrect matching could delete genuine neighbourhoods, leading to biased results.

- **Inclusion of metadata as real records**  
  Footer notes or blank rows could inflate totals or distort qualification distributions.

- **Misinterpretation of disclosure‑control notes**  
  These notes protect individuals’ privacy. Treating them as data could lead to incorrect conclusions.

- **Loss of transparency**  
  If removed rows are not shown, users cannot verify that the cleaning process was fair and accurate.

### Ethical Safeguards Used in This Step

This cleaning step includes several responsible practices:

- **Precise pattern matching**  
  The regular expression selects only valid English LSOA codes (`E01` followed by six digits), reducing the risk of accidental exclusion.

- **Preserving the original dataset**  
  The raw DataFrame remains unchanged, supporting auditability and ethical data governance.

- **Displaying counts of removed and remaining rows**  
  Supports transparency and allows verification that only non‑data rows were excluded.

- **Resetting the index**  
  Ensures clarity and prevents confusion in later analysis.

- **Checking for remaining missing values**  
  Confirms that the cleaned dataset is complete and ready for responsible

In [10]:
# Developed with assistance from ChatGPT.

# Check which rows have an English neighbourhood (LSOA) code.
# A valid code starts with E01 and has six more digits.
# Ignore spaces around codes and treat missing codes as invalid.
valid_lsoa = (
    df["mnemonic"]
    .astype("string")
    .str.strip()
    .str.fullmatch(r"E01\d{6}", na=False)
)

# Keep only rows with valid codes in a new table.
# The original table, df, stays unchanged.
df_clean = df.loc[valid_lsoa].copy()

# Restart row numbering at 0, with no gaps.
df_clean.reset_index(drop=True, inplace=True)

# Show how many rows were removed and how many remain.
# Also check whether any empty values remain.
print(f"Original rows: {len(df)}")
print(f"Rows removed: {len(df) - len(df_clean)}")
print(f"Rows remaining: {len(df_clean)}")
print(f"Columns: {df_clean.shape[1]}")
print(f"Remaining missing values: {df_clean.isna().sum().sum()}")

Original rows: 208
Rows removed: 5
Rows remaining: 203
Columns: 10
Remaining missing values: 0


### Findings

The cleaning step removed **5 non-data rows**: four source-note rows
and one blank row. The dataset now contains **203 neighbourhood
records and 10 columns**.

There are **no missing values** in the remaining records, so no
values needed to be filled in. The cleaned data is stored in
`df_clean`, while the original data remains unchanged.

---

## 5. Rename the columns

Replace the long column headings with shorter, descriptive names
that are easier to use in code.

Use `LSOA_Name` and `LSOA_Code` for the neighbourhood identifiers.
Keep the age group in `Total_Aged_16_Plus` so the population covered
by this dataset remains clear.

This step changes column names only; the values remain unchanged.

## Ethical Considerations When Renaming Qualification Column Headings

Renaming the qualification column headings is not just a technical convenience — it carries important **ethical responsibilities**, because these labels describe real educational outcomes for residents across Dudley’s neighbourhoods.  
Clear, accurate, and transparent naming helps prevent misinterpretation, supports fairness, and protects communities from being misrepresented.

### Why Ethical Care Is Needed Here

The original column headings contain:
- Detailed qualification categories  
- Age‑group definitions  
- Specific distinctions between qualification levels  
- Separate identifiers for LSOA name and code  

These headings must be preserved accurately when shortened.  
If renamed incorrectly, analysts or dashboard users may misunderstand what each column represents, leading to unfair comparisons or misleading conclusions about educational disadvantage.

### Ethical Risks if Renaming Is Done Incorrectly

If column names are shortened without care, several ethical risks arise:

- **Loss of essential meaning**  
  Removing “Level 4 qualifications or above” or “No qualifications” could distort interpretation of educational attainment.

- **Ambiguous or misleading names**  
  Vague labels may cause users to misinterpret qualification categories or population totals.

- **Inconsistent naming across datasets**  
  If `LSOA_Name` or `LSOA_Code` differ from the economic activity dataset, joins may fail or misalign, causing incorrect results.

- **Reduced transparency**  
  Poorly renamed columns make it harder for others to understand or audit the analysis.

### Ethical Safeguards Used in This Step

This renaming process includes several responsible practices:

- **Preserving essential meaning**  
  Each new name keeps the original intent clear (e.g., `Level_4_And_Above`, `No_Qualifications`).

- **Maintaining clarity for non‑technical users**  
  Shorter names improve readability in notebooks, dashboards, and reports.

- **Keeping geographic identifiers consistent**  
  `LSOA_Name` and `LSOA_Code` match the economic activity dataset, supporting fair and accurate dataset joining.

- **Retaining the age group in the population column**  
  `Total_Aged_16_Plus` ensures users understand exactly who the dataset covers.

- **Avoiding any change to underlying data values**  
  Only column labels are updated — the counts remain untouched.

### Why This Matters for Responsible Interpretation

Qualification data is sensitive and often used to:
- Identify educational disadvantage  
- Support local policy decisions  
- Allocate resources  
- Understand long‑term employment barriers  

Incorrect naming can lead to:
- Misleading conclusions  
- Stigmatisation of neighbourhoods  
- Incorrect policy insights  
- Loss of trust in the analysis  

By renaming columns carefully and transparently, this project ensures that Dudley’s qualification data is represented **fairly, accurately, and respectfully**, supporting ethical and responsible analysis throughout the workflow.



In [12]:
# Developed with assistance from ChatGPT.

# Match each original column heading to a shorter, clearer name.
# Keep the age group in the total population column name.
column_names = {
    "2021 super output area - lower layer": "LSOA_Name",
    "mnemonic": "LSOA_Code",
    "Total: All usual residents aged 16 years and over": (
        "Total_Aged_16_Plus"
    ),
    "No qualifications": "No_Qualifications",
    "Level 1 and entry level qualifications": "Level_1_And_Entry",
    "Level 2 qualifications": "Level_2",
    "Apprenticeship": "Apprenticeship",
    "Level 3 qualifications": "Level_3",
    "Level 4 qualifications or above": "Level_4_And_Above",
    "Other qualifications": "Other_Qualifications"
}

# Apply the new names without changing the data values.
# Reference: pandas documentation for renaming columns.
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html
df_clean = df_clean.rename(columns=column_names)

# Print each column name on a separate line to check the changes.
print("COLUMN NAMES:")
for column in df_clean.columns:
    print(column)

# Check that the number of rows and columns has stayed the same.
print(f"\nRows: {df_clean.shape[0]}, Columns: {df_clean.shape[1]}")

COLUMN NAMES:
LSOA_Name
LSOA_Code
Total_Aged_16_Plus
No_Qualifications
Level_1_And_Entry
Level_2
Apprenticeship
Level_3
Level_4_And_Above
Other_Qualifications

Rows: 203, Columns: 10


### Findings

The column headings now use consistent, readable names with
underscores instead of spaces.

`LSOA_Name` and `LSOA_Code` identify each neighbourhood.
`Total_Aged_16_Plus` makes clear that the qualification counts cover
residents aged 16 and over.

The dataset still contains **203 rows and 10 columns**. Only the
column headings were changed; the data values remain unchanged.

---

## 6. Check for duplicate records

Each neighbourhood should appear once in this dataset.

Check for repeated LSOA codes and completely duplicated rows.
Repeated codes could cause neighbourhoods to be counted more than
once when analysing or joining datasets.

Inspect any duplicates before deciding whether to remove them.

## Ethical Considerations When Checking for Duplicate Qualification Records

Checking for duplicate rows and repeated LSOA codes is an essential ethical step in preparing the qualifications dataset.  
Because this dataset represents real neighbourhoods in Dudley, any duplication can distort educational patterns, mislead analysis, and unfairly affect how communities are interpreted.

### Why Ethical Care Is Needed Here

Each LSOA should appear **exactly once** in the dataset.  
Qualification counts are tied to specific neighbourhoods, and duplicating a record can:

- Inflate the educational disadvantage of an area  
- Misrepresent population totals  
- Distort comparisons between neighbourhoods  
- Lead to incorrect conclusions in dashboards or reports  

Ensuring uniqueness is therefore a matter of **fairness**, **accuracy**, and **responsible data handling**.

### Ethical Risks if Duplicate Checks Are Ignored

If duplicates are not detected and addressed, several ethical risks arise:

- **Double‑counting neighbourhoods**  
  A duplicated LSOA code could cause one area to appear twice, exaggerating its population or qualification levels.

- **Misleading comparisons**  
  Analysts may believe an area has unusually high or low qualification counts when the issue is actually duplication.

- **Incorrect dataset joins**  
  Repeated LSOA codes can break merges with other datasets, causing misaligned or corrupted results.

- **Loss of transparency**  
  Without checking duplicates, users cannot trust that each neighbourhood is represented fairly.

### Ethical Safeguards Used in This Step

This duplicate‑checking process includes several responsible practices:

- **Counting fully duplicated rows**  
  Ensures no record is repeated across all columns.

- **Checking for repeated LSOA codes**  
  Confirms that each neighbourhood appears once and only once.

- **Marking all rows sharing a code**  
  Allows analysts to inspect every affected record, not just the second occurrence.

- **Displaying up to five duplicated records**  
  Supports transparency and manual verification.

- **Clear reporting**  
  The output explicitly states whether duplicates exist, supporting ethical decision‑making.

### Why This Matters for Responsible Interpretation

Qualification data is used to:

- Identify educational disadvantage  
- Support local policy decisions  
- Allocate resources fairly  
- Understand long‑term employment barriers  

Duplicate records can lead to:

- Misleading conclusions  
- Stigmatisation of neighbourhoods  
- Incorrect policy insights  
- Loss of trust in the analysis  

By checking for duplicates carefully and transparently, this project ensures that Dudley’s qualification data is handled **fairly, accurately, and respectfully**, supporting ethical and responsible analysis throughout the workflow.



In [13]:
# Developed with assistance from ChatGPT.

# Count rows that repeat an earlier row across every column.
# Reference: pandas documentation for checking duplicates.
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html
print(f"Fully duplicated rows: {df_clean.duplicated().sum()}")

# Count neighbourhood codes that appear more than once.
print(
    f"Duplicate LSOA codes: "
    f"{df_clean['LSOA_Code'].duplicated().sum()}"
)

# Mark every row sharing a code, including its first appearance.
duplicate_codes = df_clean["LSOA_Code"].duplicated(keep=False)

# Show up to five affected rows if repeated codes are found.
if duplicate_codes.any():
    print("\nROWS WITH REPEATED LSOA CODES:")
    display(df_clean.loc[duplicate_codes].head())
else:
    print("\nEach LSOA code appears only once.")

Fully duplicated rows: 0
Duplicate LSOA codes: 0

Each LSOA code appears only once.


### Findings

There are **no fully duplicated rows** and **no repeated LSOA codes**.
Each of the **203 neighbourhoods** appears once, so no records need
to be removed.

The LSOA codes are unique within this dataset and can be used to
match neighbourhoods when joining the datasets.

---

## 7. Check data types and population counts

Check how each column is stored before changing any data types.

The qualification columns contain counts of people, so their values
should be numeric, non-negative and whole numbers. Counts may
currently be stored as decimals because the original CSV included
empty cells in its footer rows.

Check for non-numeric entries, negative values and fractional values
before converting the count columns to integers.

In [14]:
# Developed with assistance from ChatGPT.

# Show the current data type of each column.
print("CURRENT DATA TYPES:")
print(df_clean.dtypes)

# Select the count columns, leaving out the area name and code.
count_columns = df_clean.columns.drop(["LSOA_Name", "LSOA_Code"])

# Check whether all count values can be read as numbers.
# Invalid entries become missing values in this temporary table.
# The cleaned dataframe is not changed.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html
numeric_counts = df_clean[count_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

# Count values that are missing or could not be read as numbers.
print("\nMISSING OR NON-NUMERIC VALUES:")
print(numeric_counts.isna().sum())

# Count values below zero in each column.
print("\nNEGATIVE VALUES:")
print((numeric_counts < 0).sum())

# Whole numbers have no remainder when divided by 1.
# Check only values that are not missing.
print("\nFRACTIONAL VALUES:")
fractional_values = (
    numeric_counts.notna()
    & numeric_counts.mod(1).ne(0)
)
print(fractional_values.sum())

CURRENT DATA TYPES:
LSOA_Name                object
LSOA_Code                object
Total_Aged_16_Plus      float64
No_Qualifications       float64
Level_1_And_Entry       float64
Level_2                 float64
Apprenticeship          float64
Level_3                 float64
Level_4_And_Above       float64
Other_Qualifications    float64
dtype: object

MISSING OR NON-NUMERIC VALUES:
Total_Aged_16_Plus      0
No_Qualifications       0
Level_1_And_Entry       0
Level_2                 0
Apprenticeship          0
Level_3                 0
Level_4_And_Above       0
Other_Qualifications    0
dtype: int64

NEGATIVE VALUES:
Total_Aged_16_Plus      0
No_Qualifications       0
Level_1_And_Entry       0
Level_2                 0
Apprenticeship          0
Level_3                 0
Level_4_And_Above       0
Other_Qualifications    0
dtype: int64

FRACTIONAL VALUES:
Total_Aged_16_Plus      0
No_Qualifications       0
Level_1_And_Entry       0
Level_2                 0
Apprenticeship          0
Leve

### Findings

The two neighbourhood identifier columns are stored as `object`,
while all eight population-count columns are stored as `float64`
(decimal numbers).

The count columns contain **no missing or non-numeric values,
no negative values and no fractional values**.

All counts are therefore suitable for conversion to integers.
This will display counts as whole numbers without changing their
values.

---

## 8. Convert population counts to integers

The previous checks confirmed that all eight count columns contain
valid, non-negative whole numbers with no missing values.

Convert these columns from `float64` to `int64` so counts display
as whole numbers, such as `1637` instead of `1637.0`.

Leave the neighbourhood names and codes unchanged.

## Ethical Considerations When Converting Qualification Counts to Integers

Converting the qualification count columns from `float64` to `int64` is a small technical step, but it carries important **ethical responsibilities**, because these values represent real residents across Dudley’s neighbourhoods.  
Ensuring that counts are valid whole numbers protects against misinterpretation and supports fair, accurate analysis.

### Why Ethical Care Is Needed Here

Qualification counts are used to understand:
- Educational attainment  
- Local skills levels  
- Barriers to employment  
- Long‑term socio‑economic disadvantage  

If these counts are incorrect, misinterpreted, or mishandled, the resulting analysis may unfairly represent neighbourhoods or lead to misleading policy insights.

### Ethical Risks if Conversion Is Done Incorrectly

If count columns are converted without proper validation, several ethical risks arise:

- **Fractional values may be silently rounded**  
  This could distort qualification totals and misrepresent communities.

- **Negative or non‑numeric values may be forced into integers**  
  Leading to inaccurate or misleading results.

- **Missing values may be converted to zeros**  
  Making some neighbourhoods appear more disadvantaged than they are.

- **Incorrect data types may break downstream analysis**  
  Causing dashboards or comparisons to display misleading information.

### Ethical Safeguards Used in This Step

This conversion process includes several responsible practices:

- **Validating all eight count columns first**  
  Ensures there are no missing, negative, fractional, or non‑numeric values.

- **Converting only after validation passes**  
  Prevents accidental corruption of the dataset.

- **Leaving LSOA names and codes unchanged**  
  Preserves geographic identifiers for accurate dataset joining.

- **Displaying updated data types and sample rows**  
  Supports transparency and allows manual verification.

- **Maintaining the original values**  
  Conversion to integers changes only the display format, not the underlying counts.

### Why This Matters for Responsible Interpretation

Qualification data is often used to:
- Identify educational disadvantage  
- Support local policy decisions  
- Allocate resources fairly  
- Understand long‑term employment barriers  

Incorrect handling of count values can lead to:
- Misleading conclusions  
- Stigmatisation of neighbourhoods  
- Incorrect policy insights  
- Loss of trust in the analysis  

By validating and converting count columns carefully and transparently, this project ensures that Dudley’s qualification data is handled **fairly, accurately, and respectfully**, supporting ethical and responsible analysis throughout the workflow.



In [15]:
# Developed with assistance from ChatGPT.

# Convert the checked population-count columns to whole numbers.
# Reference: pandas documentation for changing data types.
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.astype.html
df_clean[count_columns] = df_clean[count_columns].astype("int64")

# Check that the count columns now use the integer data type.
print("UPDATED DATA TYPES:")
print(df_clean.dtypes)

# Display five rows to check how the counts now appear.
print("\nFIRST FIVE ROWS:")
display(df_clean.head())

UPDATED DATA TYPES:
LSOA_Name               object
LSOA_Code               object
Total_Aged_16_Plus       int64
No_Qualifications        int64
Level_1_And_Entry        int64
Level_2                  int64
Apprenticeship           int64
Level_3                  int64
Level_4_And_Above        int64
Other_Qualifications     int64
dtype: object

FIRST FIVE ROWS:


,LSOA_Name,LSOA_Code,Total_Aged_16_Plus,No_Qualifications,Level_1_And_Entry,Level_2,Apprenticeship,Level_3,Level_4_And_Above,Other_Qualifications
0,Dudley 001A,E01009763,1637,436,183,289,102,286,296,45
1,Dudley 001B,E01009764,1266,306,132,187,98,214,279,50
2,Dudley 001C,E01009767,1265,358,146,220,73,181,238,49
3,Dudley 001D,E01009769,1428,467,157,244,88,211,221,40
4,Dudley 001E,E01009770,1215,390,152,188,72,181,198,34


### Findings

All eight population-count columns now use the `int64` data type,
so counts are stored as whole numbers without decimal places.

The conversion preserved the original count values.
`LSOA_Name` and `LSOA_Code` remain unchanged as `object` columns.

---

## 9. Compare qualification counts with population totals

Add the seven qualification categories for each neighbourhood and
compare their sum with `Total_Aged_16_Plus`.

This checks whether the category counts match the recorded population.
Any differences will be inspected before deciding whether further
action is needed. No values will be changed during this check.

## Ethical Considerations When Comparing Qualification Sums with Population Totals

This step checks whether the seven qualification categories for each neighbourhood add up to the recorded population aged 16 years and over.  
Although this appears to be a simple arithmetic check, it carries important **ethical responsibilities**, because qualification data reflects real educational outcomes for residents across Dudley’s neighbourhoods.

### Why Ethical Care Is Needed Here

Qualification counts are used to understand:
- Educational attainment  
- Skills levels  
- Barriers to employment  
- Long‑term socio‑economic disadvantage  

If the qualification categories do not sum to the population total, it may indicate:
- Data quality issues  
- Misclassification of residents  
- Missing categories  
- Disclosure‑control adjustments  
- Export or formatting errors  

These issues can unfairly distort how communities are perceived.

### Ethical Risks if Totals Are Not Checked

If this comparison is skipped or handled incorrectly, several ethical risks arise:

- **Misrepresenting neighbourhoods**  
  A mismatch may suggest lower or higher qualification levels than reality.

- **Incorrect policy insights**  
  Decisions based on inaccurate totals may misallocate resources.

- **Misleading dashboards**  
  Visualisations may show inconsistent or contradictory values.

- **Unfair comparisons**  
  Neighbourhoods may appear disadvantaged or advantaged due to data inconsistencies rather than real conditions.

- **Loss of transparency**  
  Users may not understand why qualification totals differ from population counts.

### Ethical Safeguards Used in This Step

This comparison process includes several responsible practices:

- **Summing qualification categories explicitly**  
  Ensures clarity and avoids relying on assumptions about dataset structure.

- **Comparing sums with the recorded population total**  
  Provides a direct integrity check for each neighbourhood.

- **Reporting matching and non‑matching totals**  
  Supports transparency and helps identify areas requiring further investigation.

- **Displaying examples of mismatches**  
  Allows analysts to inspect potential issues manually before deciding on any corrective action.

- **Avoiding changes to the underlying data**  
  No values are altered during this check, preserving the dataset’s integrity.

### Why This Matters for Responsible Interpretation

Qualification data is often used to:
- Identify educational disadvantage  
- Support local policy decisions  
- Allocate resources fairly  
- Understand long‑term employment barriers  

Incorrect handling of category totals can lead to:
- Misleading conclusions  
- Stigmatisation of neighbourhoods  
- Incorrect policy insights  
- Loss of trust in the analysis  

By comparing qualification sums with population totals carefully and transparently, this project ensures that Dudley’s qualification data is handled **fairly, accurately, and respectfully**, supporting ethical and responsible analysis throughout the workflow.



In [17]:
# Developed with assistance from ChatGPT.

# List the seven qualification categories, excluding the total.
qualification_columns = [
    "No_Qualifications",
    "Level_1_And_Entry",
    "Level_2",
    "Apprenticeship",
    "Level_3",
    "Level_4_And_Above",
    "Other_Qualifications"
]

# Add the qualification counts across each row.
# axis=1 means add across columns for each neighbourhood.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sum.html
qualification_sum = df_clean[qualification_columns].sum(axis=1)

# Compare the calculated sum with the recorded population total.
# A difference of zero means the totals match.
# Developed with assistance from ChatGPT.
difference = qualification_sum - df_clean["Total_Aged_16_Plus"]

print(f"Neighbourhoods checked: {len(df_clean)}")
print(f"Matching totals: {(difference == 0).sum()}")
print(f"Non-matching totals: {(difference != 0).sum()}")
print(f"Largest absolute difference: {difference.abs().max()}")

# If any totals differ, display up to five examples.
# Developed with assistance from ChatGPT.
if (difference != 0).any():
    total_checks = df_clean[
        ["LSOA_Name", "LSOA_Code", "Total_Aged_16_Plus"]
    ].copy()
    total_checks["Qualification_Sum"] = qualification_sum
    total_checks["Difference"] = difference

    print("\nEXAMPLES OF NON-MATCHING TOTALS:")
    display(total_checks.loc[difference != 0].head())

Neighbourhoods checked: 203
Matching totals: 203
Non-matching totals: 0
Largest absolute difference: 0


### Findings

The seven qualification categories add up to `Total_Aged_16_Plus`
for **all 203 neighbourhoods**.

There are **no mismatched totals**, and the largest absolute
difference is **0**. No corrections are needed following this check.

---

## 10. Save the cleaned dataset

Save the 203 cleaned neighbourhood records to
`cleaned_datasets/dudley_qualifications_cleaned.csv`.

Exclude the dataframe index from the CSV and keep the original
raw dataset unchanged.

## Ethical Considerations When Saving the Cleaned Qualifications Dataset

Saving the cleaned qualifications dataset is a crucial ethical step in the workflow.  
This dataset represents real residents across Dudley, and preserving its accuracy, structure, and integrity is essential for responsible analysis, fair interpretation, and transparent data governance.

### Why Ethical Care Is Needed Here

Once a dataset is cleaned and validated, it becomes the foundation for:

- Visualisations  
- Statistical summaries  
- Cross‑dataset joins  
- Public‑facing dashboards  
- Policy‑relevant insights  

If the saved dataset contains errors, missing values, or unintended changes, the consequences can cascade through the entire project and lead to **misleading or unfair conclusions** about neighbourhoods.

### Ethical Risks if Saving Is Done Incorrectly

If the dataset is saved or reloaded incorrectly, several ethical risks arise:

- **Loss of data integrity**  
  Exporting may accidentally drop rows, alter values, or change column types.

- **Silent introduction of missing values**  
  Encoding issues or formatting errors may convert valid values to `NaN`.

- **Duplicate or corrupted records**  
  Incorrect saving may introduce repeated rows or misaligned identifiers.

- **Misalignment of LSOA codes**  
  If identifiers change during export, dataset joins may fail or misrepresent communities.

- **Reduced transparency**  
  Without verification, users cannot be confident that the saved dataset matches the cleaned version.

### Ethical Safeguards Used in This Step

This saving process includes several responsible practices:

- **Creating a dedicated `cleaned_datasets` folder**  
  Keeps cleaned data separate from raw data, supporting transparency and auditability.

- **Saving without the DataFrame index**  
  Prevents accidental introduction of non‑data columns that could confuse downstream users.

- **Using UTF‑8‑SIG encoding**  
  Ensures that special characters in LSOA names are preserved correctly.

- **Preserving the raw dataset unchanged**  
  Maintains a clear audit trail and supports ethical data governance.

- **Confirming the save location and dataset dimensions**  
  Provides immediate feedback that the export was successful and complete.

### Why This Matters for Responsible Interpretation

Qualification data is sensitive and often used to:

- Identify educational disadvantage  
- Support local policy decisions  
- Allocate resources fairly  
- Understand long‑term employment barriers  

Errors introduced during saving can lead to:

- Misleading conclusions  
- Stigmatisation of neighbourhoods  
- Incorrect policy insights  
- Loss of trust in the analysis  

By saving and verifying the cleaned dataset carefully, this project ensures that Dudley’s qualification data is preserved **fairly, accurately, and respectfully**, supporting ethical and responsible analysis throughout the workflow.



In [18]:
# Developed with assistance from ChatGPT.

from pathlib import Path

# Create a cleaned_datasets folder beside the raw_datasets folder.
cleaned_folder = Path(raw_path).parent.parent / "cleaned_datasets"
cleaned_folder.mkdir(parents=True, exist_ok=True)

# Set the filename for the cleaned qualifications dataset.
cleaned_path = cleaned_folder / "dudley_qualifications_cleaned.csv"

# Save without adding the dataframe's row numbers as a column.
# Use UTF-8 encoding for the CSV.
# Reference: pandas documentation for saving CSV files.
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html
df_clean.to_csv(
    cleaned_path,
    index=False,
    encoding="utf-8-sig"
)

# Confirm the save location and dataset dimensions.
print(f"Saved to: {cleaned_path}")
print(f"Rows: {df_clean.shape[0]}, Columns: {df_clean.shape[1]}")

Saved to: cleaned_datasets\dudley_qualifications_cleaned.csv
Rows: 203, Columns: 10


### Cleaning summary

The cleaned qualifications dataset contains **203 neighbourhoods
and 10 columns**.

- Removed four source-note rows and one blank row.
- Renamed columns for clarity and consistency.
- Confirmed no missing values or duplicate LSOA codes.
- Checked that all population counts are non-negative whole numbers.
- Converted the eight count columns to integers.
- Confirmed qualification categories match the population total
  for every neighbourhood.

Saved as `cleaned_datasets/dudley_qualifications_cleaned.csv`.
The original raw dataset remains unchanged.

---

## Conclusion

The qualifications dataset has been cleaned and saved with
**203 unique Dudley neighbourhoods and 10 columns**.

Five non-data rows were removed, column names were standardised,
and population counts were converted to integers. Validation confirmed
no missing values, duplicate LSOA codes, negative counts or fractional
values. Qualification categories match the recorded population total
for all 203 neighbourhoods.

The cleaned dataset is saved as
`cleaned_datasets/dudley_qualifications_cleaned.csv`.
The original raw dataset remains unchanged.

## Ethical Considerations in the Final Conclusions

The cleaning and validation of the qualifications dataset is now complete, and all checks confirm that the data is reliable, consistent, and ethically suitable for further analysis.  
Because this dataset represents real residents across Dudley’s neighbourhoods, the conclusions must reflect not only technical correctness but also the **ethical responsibilities** upheld throughout the process.

### Ethical Significance of the Cleaning Outcome

The final cleaned dataset contains:

- 203 verified neighbourhood (LSOA) records  
- No missing values  
- No duplicate rows  
- No repeated LSOA codes  
- Seven qualification categories that **perfectly match** the population total for every neighbourhood  
- All qualification counts stored as whole numbers (`int64`)  
- Preserved and clearly labelled geographic identifiers (`LSOA_Name`, `LSOA_Code`)  

These results demonstrate that the dataset is structurally sound, internally consistent, and ethically robust.

### Ethical Risks Avoided Through Careful Cleaning

The workflow avoided several potential harms:

- **Misrepresenting neighbourhoods** through missing or duplicated records  
- **Inflating or deflating qualification totals** due to incorrect data types  
- **Misaligned geographic identifiers** that could distort cross‑dataset joins  
- **Silent errors during export** that could mislead dashboards or reports  
- **Unfair comparisons** caused by inconsistent or corrupted values  

Avoiding these risks is essential when analysing educational attainment, skills levels, and long‑term employment barriers.

### Ethical Strengths of the Completed Workflow

The cleaning process demonstrates:

- **Transparency** — every transformation was documented and justified  
- **Accuracy** — all values were validated before conversion  
- **Fairness** — no LSOA was removed, duplicated, or misrepresented  
- **Accountability** — the saved dataset was verified against the cleaned DataFrame  
- **Respect for communities** — the raw dataset remains unchanged and auditable  
- **Consistency** — column names are clear, meaningful, and aligned with other datasets  

These practices ensure that the dataset can be used responsibly in later notebooks, dashboards, and reports.

### Ethical Conclusion

The cleaned qualifications dataset is now ethically robust and technically reliable.  
It preserves the integrity of Dudley’s LSOA‑level information, supports fair and accurate analysis, and provides a trustworthy foundation for exploring educational attainment and employment barriers without misrepresenting or harming the communities described.

